# Wave Gate Check

Deterministic gate for the build waves job. Runs after each `wN_integrate` Genie Code task.

A Genie Code task can finish "successfully" even when it concluded the wave failed, so this notebook reads the integrator's gate file and sets a `gate_passed` **task value**. A downstream `If/else` condition task checks the value — if false, all remaining wave tasks are `Excluded` (zero compute on cascade).

Gate file: `<build_root>/<status_dir>/<run_id>/wave_<wave>_gate.json` (written by the wave integrator skill).

**Runtime gate.** If the wave lists `runtime_gate` targets, this notebook also checks each one itself (it does not trust Genie Code reports):
* **app**: active deployment `SUCCEEDED` and created after the build run started, app `RUNNING` / compute `ACTIVE`, unchanged through `soak_minutes`, OTel error lines for the deployment ≤ `max_errors`, all `required_markers` present.
* **job**: latest run since the build run started finished `SUCCESS`.
* **pipeline**: latest update since the build run started `COMPLETED` (or stays `RUNNING` through the soak).

The runtime verdict is written to `wave_<wave>_runtime.json` next to the gate file. Runtime checks are skipped if the static gate already failed.

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("wave", "", "Wave id")
dbutils.widgets.text("run_id", "", "Job run id")
dbutils.widgets.text("build_root", "", "Build root (project Git folder)")
dbutils.widgets.text("manifest_path", "", "Manifest path (waves.json)")

wave = dbutils.widgets.get("wave").strip()
run_id = dbutils.widgets.get("run_id").strip()
build_root = dbutils.widgets.get("build_root").strip()
manifest_path = dbutils.widgets.get("manifest_path").strip()

if not wave or not run_id:
    raise ValueError(f"wave and run_id are required (got wave={wave!r}, run_id={run_id!r})")

print(f"wave={wave} run_id={run_id}")

In [0]:
import json
import os

with open(manifest_path) as f:
    manifest = json.load(f)

wave_cfg = next((w for w in manifest["waves"] if w["wave_id"] == wave), None)
if wave_cfg is None:
    raise ValueError(f"Wave {wave!r} not found in {manifest_path}")

status_dir = os.path.join(build_root, manifest["status_dir"], run_id)
problems = []

# 1. Integrator gate file
gate_path = os.path.join(status_dir, f"wave_{wave}_gate.json")
gate = None
if not os.path.exists(gate_path):
    problems.append(f"gate file missing: {gate_path}")
else:
    with open(gate_path) as f:
        gate = json.load(f)
    if gate.get("status") != "pass":
        problems.append(f"gate status={gate.get('status')!r}: {gate.get('reason', '')}")
    if gate.get("stray_files"):
        problems.append(f"stray files: {gate['stray_files']}")
    # Carryover from earlier runs is tolerated, except code nobody claimed by the last wave.
    if manifest["waves"][-1]["wave_id"] == wave:
        unclaimed = [c.get("path") for c in (gate.get("carryover_files") or [])
                     if c.get("kind") in ("prior_run_code", "owned_by_later_wave")]
        if unclaimed:
            problems.append(f"unclaimed carryover in last wave: {unclaimed}")

# 2. Independent check of every session status file (does not trust the integrator alone)
for s in wave_cfg["sessions"]:
    p = os.path.join(status_dir, f"{s['session_id']}.json")
    if not os.path.exists(p):
        problems.append(f"session {s['session_id']}: status file missing")
        continue
    with open(p) as f:
        st = json.load(f)
    if st.get("status") != "pass":
        problems.append(
            f"session {s['session_id']}: status={st.get('status')!r} {st.get('blocking', '')}"
        )

# 3. Converge sessions: a 'pass' must end with a stable attempt
for s in wave_cfg["sessions"]:
    if s.get("mode") != "converge":
        continue
    p = os.path.join(status_dir, f"{s['session_id']}.json")
    if not os.path.exists(p):
        continue  # already reported above
    with open(p) as f:
        st = json.load(f)
    attempts = st.get("attempts") or []
    if st.get("status") == "pass" and (not attempts or attempts[-1].get("result") != "stable"):
        problems.append(f"session {s['session_id']}: status=pass but last attempt is not 'stable'")

static_problems = list(problems)
print(f"static gate: {'PASS' if not static_problems else 'FAIL'} ({len(static_problems)} problems)")

In [0]:
import re
import time
from datetime import datetime, timezone

from databricks.sdk import WorkspaceClient

DEFAULT_ERROR_PATTERNS = [
    r"error TS\d+",
    r"ELIFECYCLE",
    r"Error building app",
    r"Unhandled",
    r"UnhandledPromiseRejection",
    r"FATAL",
]
_TABLE_RE = re.compile(r"^[A-Za-z0-9_`\-]+\.[A-Za-z0-9_`\-]+\.[A-Za-z0-9_`\-]+$")
_ATTR_RE = re.compile(r"^[A-Za-z0-9_.\-]+$")

runtime_names = wave_cfg.get("runtime_gate") or []
runtime_defs = manifest.get("runtime_targets") or {}
runtime_results = {}


def _ts(v):
    """Parse an ISO string or epoch-millis into an aware UTC datetime."""
    if v in (None, ""):
        return None
    if isinstance(v, (int, float)):
        return datetime.fromtimestamp(v / 1000, tz=timezone.utc)
    return datetime.fromisoformat(str(v).replace("Z", "+00:00"))


def _enum(v):
    return getattr(v, "value", v)


def check_app_logs(tgt, app_name, dep_id, dep_created):
    """Count error lines and check required markers for one app deployment."""
    tel = tgt.get("telemetry") or {}
    table = tel.get("logs_table")
    out = {"logs_table": table, "error_count": None, "error_samples": [], "missing_markers": []}
    if not table:
        out["note"] = "no telemetry.logs_table: log checks skipped"
        return out, []
    dep_attr = tel.get("deployment_id_attribute", "app.deployment_id")
    src_attr = tel.get("log_source_attribute", "app.log_source")
    if not _TABLE_RE.match(table) or not _ATTR_RE.match(dep_attr) or not _ATTR_RE.match(src_attr):
        return out, [f"invalid telemetry identifiers: {table!r} / {dep_attr!r} / {src_attr!r}"]

    # Rows for this deployment (build + runtime), plus same-service rows after it was created
    # in case runtime lines don't carry the deployment id attribute.
    rows = spark.sql(
        f"""
        SELECT time, severity_text, body::string AS body, attributes:['{src_attr}']::string AS src
        FROM {table}
        WHERE date >= date(:since)
          AND (attributes:['{dep_attr}']::string = :dep
               OR (service_name = :svc AND time >= :since_ts))
        ORDER BY time
        LIMIT 50000
        """,
        args={"since": dep_created.date().isoformat(), "since_ts": dep_created, "dep": dep_id, "svc": app_name},
    ).collect()

    err_res = [re.compile(p) for p in (tel.get("error_patterns") or DEFAULT_ERROR_PATTERNS)]
    ign_res = [re.compile(p) for p in (tel.get("ignore_patterns") or [])]
    errors = []
    for r in rows:
        body = r["body"] or ""
        if any(p.search(body) for p in ign_res):
            continue
        if (r["severity_text"] or "").upper() in ("ERROR", "FATAL") or any(p.search(body) for p in err_res):
            errors.append(f"{r['time']} [{r['src'] or '?'}] {body[:200]}")

    runtime_rows = [r for r in rows if (r["src"] or "").upper() != "BUILD"]
    missing = [m for m in (tel.get("required_markers") or [])
               if not any(re.search(m, r["body"] or "") for r in runtime_rows)]

    out.update({"rows": len(rows), "runtime_rows": len(runtime_rows), "error_count": len(errors),
                "error_samples": errors[:10], "missing_markers": missing})
    max_errors = int((tgt.get("stability") or {}).get("max_errors", 0))
    probs = []
    if len(errors) > max_errors:
        probs.append(f"{len(errors)} error log lines (max {max_errors}); first: {errors[0] if errors else ''}")
    if missing:
        probs.append(f"required log markers not found: {missing}")
    return out, probs


def check_app(tgt, run_start):
    name = tgt["name"]
    soak = int((tgt.get("stability") or {}).get("soak_minutes", 10))
    fresh = tgt.get("require_fresh_deployment", True)
    info, probs = {"app": name}, []

    app = w.apps.get(name)
    ad, pd = app.active_deployment, app.pending_deployment
    if pd is not None and _enum(getattr(pd.status, "state", None)) == "IN_PROGRESS":
        probs.append(f"pending deployment {pd.deployment_id} still IN_PROGRESS")
    if ad is None:
        return info, probs + ["no active deployment"]
    dep_id, dep_created = ad.deployment_id, _ts(ad.create_time)
    dstate = _enum(getattr(ad.status, "state", None))
    info.update({"deployment_id": dep_id, "deployment_state": dstate, "deployment_created": str(dep_created)})
    if dstate != "SUCCEEDED":
        probs.append(f"active deployment {dep_id} is {dstate}: {getattr(ad.status, 'message', '')}")
    if fresh and run_start and dep_created and dep_created < run_start:
        probs.append(f"active deployment {dep_id} ({dep_created}) predates this build run ({run_start})")

    def _states(a):
        return _enum(getattr(a.app_status, "state", None)), _enum(getattr(a.compute_status, "state", None))

    app_state, compute_state = _states(app)
    info.update({"app_state": app_state, "compute_state": compute_state})
    if app_state != "RUNNING" or compute_state != "ACTIVE":
        probs.append(f"app_status={app_state} compute_status={compute_state}: {getattr(app.app_status, 'message', '')}")

    # Soak only when everything so far is healthy
    if not probs and soak > 0:
        print(f"  {name}: soaking {soak} min on deployment {dep_id} ...")
        deadline = time.time() + soak * 60
        while time.time() < deadline:
            time.sleep(60)
            a = w.apps.get(name)
            s_app, s_comp = _states(a)
            cur_dep = a.active_deployment.deployment_id if a.active_deployment else None
            if s_app != "RUNNING" or s_comp != "ACTIVE" or cur_dep != dep_id:
                probs.append(f"unstable during soak: app_status={s_app} compute={s_comp} deployment={cur_dep}")
                break
        info["soak_minutes"] = soak

    if dep_created:
        log_info, log_probs = check_app_logs(tgt, name, dep_id, dep_created)
        info["logs"] = log_info
        probs += log_probs
    return info, probs


def check_job(tgt, run_start):
    info, probs = {}, []
    job_id = tgt.get("id")
    if not job_id:
        matches = list(w.jobs.list(name=tgt["name"]))
        if len(matches) != 1:
            return info, [f"job name {tgt['name']!r} matched {len(matches)} jobs; set runtime target 'id'"]
        job_id = matches[0].job_id
    info["job_id"] = job_id
    soak = int((tgt.get("stability") or {}).get("soak_minutes", 10))
    start_ms = int(run_start.timestamp() * 1000) if run_start else None
    deadline = time.time() + soak * 60
    while True:
        runs = list(w.jobs.list_runs(job_id=int(job_id), start_time_from=start_ms, limit=5))
        if not runs:
            return info, ["no job run started since the build run began"]
        latest = runs[0]
        life = _enum(getattr(latest.state, "life_cycle_state", None))
        if life in ("TERMINATED", "SKIPPED", "INTERNAL_ERROR") or time.time() > deadline:
            break
        time.sleep(30)
    result = _enum(getattr(latest.state, "result_state", None))
    info.update({"run_id": latest.run_id, "life_cycle_state": life, "result_state": result})
    if result != "SUCCESS":
        probs.append(f"latest run {latest.run_id}: {life}/{result} {getattr(latest.state, 'state_message', '')}")
    return info, probs


def check_pipeline(tgt, run_start):
    info, probs = {}, []
    pid = tgt.get("id")
    if not pid:
        matches = [p for p in w.pipelines.list_pipelines() if p.name == tgt["name"]]
        if len(matches) != 1:
            return info, [f"pipeline name {tgt['name']!r} matched {len(matches)} pipelines; set runtime target 'id'"]
        pid = matches[0].pipeline_id
    info["pipeline_id"] = pid
    soak = int((tgt.get("stability") or {}).get("soak_minutes", 10))
    start_ms = int(run_start.timestamp() * 1000) if run_start else 0
    deadline = time.time() + soak * 60
    while True:
        ups = [u for u in (w.pipelines.list_updates(pipeline_id=pid, max_results=10).updates or [])
               if (u.creation_time or 0) >= start_ms]
        if not ups:
            return info, ["no pipeline update started since the build run began"]
        latest = max(ups, key=lambda u: u.creation_time or 0)
        state = _enum(latest.state)
        if state in ("COMPLETED", "FAILED", "CANCELED") or time.time() > deadline:
            break
        time.sleep(30)
    info.update({"update_id": latest.update_id, "state": state})
    if state == "RUNNING":
        info["note"] = "continuous pipeline stayed RUNNING through the soak"
    elif state != "COMPLETED":
        probs.append(f"latest update {latest.update_id} is {state}")
    return info, probs


if not runtime_names:
    print("no runtime_gate targets for this wave")
elif static_problems:
    print("static gate failed: runtime checks skipped")
    runtime_results = {n: {"skipped": "static gate failed"} for n in runtime_names}
else:
    w = WorkspaceClient()
    try:
        run_start = _ts(w.jobs.get_run(int(run_id)).start_time)
    except Exception as e:  # interactive run with a fake run_id
        print(f"WARN: could not read build run start time ({e}); freshness check disabled")
        run_start = None
    checkers = {"app": check_app, "job": check_job, "pipeline": check_pipeline}
    for n in runtime_names:
        tgt = runtime_defs.get(n)
        if not tgt:
            problems.append(f"runtime target {n!r}: not defined in manifest runtime_targets")
            continue
        fn = checkers.get(tgt.get("type"))
        if not fn:
            problems.append(f"runtime target {n!r}: unsupported type {tgt.get('type')!r}")
            continue
        try:
            info, probs = fn(tgt, run_start)
        except Exception as e:
            info, probs = {}, [f"check raised {type(e).__name__}: {e}"]
        info["status"] = "pass" if not probs else "fail"
        info["problems"] = probs
        runtime_results[n] = info
        problems += [f"runtime {n}: {p}" for p in probs]
        print(f"  runtime {n} ({tgt.get('type')}): {info['status'].upper()} {probs or ''}")

if runtime_names:
    os.makedirs(status_dir, exist_ok=True)
    with open(os.path.join(status_dir, f"wave_{wave}_runtime.json"), "w") as f:
        json.dump({"wave": wave, "run_id": run_id, "targets": runtime_results}, f, indent=2, default=str)

In [0]:
if gate:
    skipped = [c["name"] for c in gate.get("checks", []) if c.get("result") == "skipped"]
    print(f"commit_sha: {gate.get('commit_sha') or '(none)'}")
    print(f"skipped checks: {skipped or 'none'}")
    if gate.get("suppressions"):
        print(f"suppressions to review: {json.dumps(gate['suppressions'], indent=2)}")
    if gate.get("requests"):
        print(f"requests for next wave / user: {json.dumps(gate['requests'], indent=2)}")
    if gate.get("carryover_files"):
        print(f"carryover from earlier runs (not strays): {json.dumps(gate['carryover_files'], indent=2)}")

passed = len(problems) == 0

# Set task value for the downstream If/else condition task.
# The condition_task checks: {{tasks.wN_gate.values.gate_passed}} == "true"
# If false, all downstream wave tasks are Excluded (zero-compute cascade stop).
try:
    dbutils.jobs.taskValues.set(key="gate_passed", value="true" if passed else "false")
except Exception:
    pass  # interactive run — no job context, taskValues unavailable

result = {
    "wave": wave,
    "status": "pass" if passed else "fail",
    "commit_sha": (gate or {}).get("commit_sha", ""),
    "runtime": {n: r.get("status", r.get("skipped")) for n, r in runtime_results.items()},
}
if problems:
    result["problems"] = problems
    print(f"Wave {wave} gate FAILED:\n  - " + "\n  - ".join(problems))
else:
    print(f"Wave {wave} gate PASSED ({len(wave_cfg['sessions'])} sessions, {len(runtime_results)} runtime targets).")

# Exit with the payload but do NOT raise — the If/else task handles the cascade.
dbutils.notebook.exit(json.dumps(result, default=str))